## Implementation of DGCNN (Model-1) 

In [9]:
# Dependencies & Constants

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Flatten, Layer, LSTM, Embedding, Lambda, ReLU
from tensorflow.keras.utils import plot_model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

NUM_API_CALLS = 307
SEQUENCE_LENGTH = 100

In [10]:
# An implementation of the Graph Convolution layer from the paper
# TODO: May have some errors, will re-check later

class GraphConvLayer(Layer):
    """Custom Graph Convolutional Layer."""
    def __init__(self, output_dim, **kwargs):
        super(GraphConvLayer, self).__init__(**kwargs)
        self.output_dim = output_dim

    def build(self, input_shape):
        self.kernel = self.add_weight(name='kernel',
                                      shape=(input_shape[1][-1], self.output_dim),
                                      initializer='glorot_uniform',
                                      trainable=True)
        super(GraphConvLayer, self).build(input_shape)

    def call(self, inputs):
        adj_matrix, features = inputs
        support = tf.matmul(features, adj_matrix)
        output = tf.matmul(support, self.kernel)
        return output

    def get_config(self):
        config = super().get_config()
        config.update({"output_dim": self.output_dim})
        return config

In [11]:
# Model builder helper function

def build_dgcnn_model(seq_len=SEQUENCE_LENGTH, num_apis=NUM_API_CALLS, conv_units=32, dense_units=128):
    """Builds the complete, fully Keras-compatible DGCNN model architecture."""
    input_graph = Input(shape=(num_apis, num_apis), name='graph_input')
    input_sequence = Input(shape=(seq_len,), name='sequence_input')
    
    features = Lambda(
        lambda x: tf.one_hot(tf.cast(x, tf.int32), depth=num_apis),
        name='one_hot_encoding'
    )(input_sequence)
    
    graph_conv_output = GraphConvLayer(output_dim=conv_units)([input_graph, features])
    
    graph_conv_output_activated = ReLU()(graph_conv_output)
    
    flattened = Flatten()(graph_conv_output_activated)
    dense_layer = Dense(dense_units, activation='relu')(flattened)
    output = Dense(1, activation='sigmoid', name='output')(dense_layer)
    
    model = Model(inputs=[input_graph, input_sequence], outputs=output)
    model.compile(optimizer='adam',
                  loss='binary_crossentropy',
                  metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])
    return model


In [12]:
# Build the model

dgcnn_model = build_dgcnn_model()
print("\n--- DGCNN Model Architecture ---")
dgcnn_model.summary()


--- DGCNN Model Architecture ---
Model: "model_2"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 sequence_input (InputLayer  [(None, 100)]                0         []                            
 )                                                                                                
                                                                                                  
 graph_input (InputLayer)    [(None, 307, 307)]           0         []                            
                                                                                                  
 one_hot_encoding (Lambda)   (None, 100, 307)             0         ['sequence_input[0][0]']      
                                                                                                  
 graph_conv_layer_2 (GraphC  (None, 100, 32)              

In [ ]:

import time
import logging
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

EPOCHS = 20
BATCH_SIZE = 64

def plot_training_history(history, model_name):
    """Visualizes the model's training and validation performance."""
    plt.figure(figsize=(14, 6))
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Training Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.title(f'{model_name} - Accuracy', fontsize=14)
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Training Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title(f'{model_name} - Loss', fontsize=14)
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.suptitle(f'Training History for {model_name}', fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.savefig(f'{model_name}_training_history.png')
    plt.show()

def plot_confusion_matrix_and_roc(y_true, y_pred, y_prob, model_name):
    """Visualizes the confusion matrix and ROC curve for model evaluation."""
    plt.figure(figsize=(16, 7))
    plt.subplot(1, 2, 1)
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Goodware', 'Malware'], yticklabels=['Goodware', 'Malware'])
    plt.title(f'{model_name} - Confusion Matrix', fontsize=14)
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.subplot(1, 2, 2)
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'{model_name} - Receiver Operating Characteristic (ROC)', fontsize=14)
    plt.legend(loc="lower right")
    plt.suptitle(f'Evaluation Metrics for {model_name}', fontsize=16)
    plt.savefig(f'{model_name}_evaluation_plots.png')
    plt.show()

def train_and_evaluate(model, model_name, X_train, y_train, X_test, y_test):
    """A comprehensive function to train a model, evaluate its performance, and generate visualizations."""
    logging.info(f"--- Starting Training for {model_name} ---")
    early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
    start_time = time.time()
    history = model.fit(
        X_train, y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_split=0.2,
        callbacks=[early_stopping],
        verbose=1
    )
    training_time = time.time() - start_time
    logging.info(f"Training for {model_name} completed in {training_time:.2f} seconds.")
    logging.info(f"--- Evaluating {model_name} on the Test Set ---")
    
    if isinstance(X_test, list):
      y_pred_prob = model.predict(X_test).ravel()
    else:
      y_pred_prob = model.predict(X_test).ravel()
      
    y_pred_class = (y_pred_prob > 0.5).astype(int)

    print(f"\n--- Classification Report for {model_name} ---")
    print(classification_report(y_test, y_pred_class, target_names=['Goodware', 'Malware']))
    
    plot_training_history(history, model_name)
    plot_confusion_matrix_and_roc(y_test, y_pred_class, y_pred_prob, model_name)

    return model

In [14]:
# Load the data from the preprocessor scripts

import numpy
import pandas

data_balanced = pandas.read_csv("../data/processed/balanced_dataset.csv")

X_balanced = data_balanced.drop(["malware"], axis=1)
y_balanced = data_balanced["malware"]

X_graphs_balanced = numpy.load("../data/graphs/balanced_graphs.npy")

In [15]:
# Use the data we preprocessed earlier and fit it into the model

X_train_seq_b, X_test_seq_b, y_train_b, y_test_b = train_test_split(
    X_balanced, y_balanced, test_size=0.2, random_state=42, stratify=y_balanced
)    

X_train_graph_b, X_test_graph_b, _, _ = train_test_split(
    X_graphs_balanced, y_balanced, test_size=0.2, random_state=42, stratify=y_balanced
)
    
dgcnn_balanced_model = build_dgcnn_model()
train_and_evaluate(
    dgcnn_balanced_model, "DGCNN_(Balanced)",
    [X_train_graph_b, X_train_seq_b], y_train_b,
    [X_test_graph_b, X_test_seq_b], y_test_b
)

Epoch 1/20
22/22 [==============================] - 2s 64ms/step - loss: 0.4273 - accuracy: 0.8000 - auc: 0.8994 - val_loss: 0.2456 - val_accuracy: 0.9191 - val_auc: 0.9767
Epoch 2/20
22/22 [==============================] - 1s 47ms/step - loss: 0.1603 - accuracy: 0.9442 - auc: 0.9875 - val_loss: 0.1207 - val_accuracy: 0.9624 - val_auc: 0.9932
Epoch 3/20
22/22 [==============================] - 1s 47ms/step - loss: 0.0574 - accuracy: 0.9891 - auc: 0.9994 - val_loss: 0.0829 - val_accuracy: 0.9711 - val_auc: 0.9983
Epoch 4/20
22/22 [==============================] - 1s 42ms/step - loss: 0.0229 - accuracy: 0.9978 - auc: 1.0000 - val_loss: 0.0501 - val_accuracy: 0.9827 - val_auc: 0.9989
Epoch 5/20
22/22 [==============================] - 1s 42ms/step - loss: 0.0103 - accuracy: 1.0000 - auc: 1.0000 - val_loss: 0.0393 - val_accuracy: 0.9884 - val_auc: 0.9994
Epoch 6/20
22/22 [==============================] - 1s 43ms/step - loss: 0.0058 - accuracy: 1.0000 - auc: 1.0000 - val_loss: 0.0343 - v